<a href="https://colab.research.google.com/github/bushrahaji412-lab/MY_ML_INTERNSHIP/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bushrahaji412-lab/MY_ML_INTERNSHIP/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [52]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [53]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
BASE = "hf://datasets/FlyRank/internship-warehouse"

print("Connected!")

Connected!


In [54]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/fact_content_daily_performance/**/*.parquet') LIMIT 5")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [55]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/fact_content_daily_performance/**/*.parquet') LIMIT 5").df().to_string()

'                 column_name column_type null   key default extra\n0                report_date        DATE  YES  None    None  None\n1             client_hash_id     VARCHAR  YES  None    None  None\n2            content_hash_id     VARCHAR  YES  None    None  None\n3             client_has_gsc     BOOLEAN  YES  None    None  None\n4             client_has_ga4     BOOLEAN  YES  None    None  None\n5         gsc_data_available     BOOLEAN  YES  None    None  None\n6         ga4_data_available     BOOLEAN  YES  None    None  None\n7            gsc_impressions      BIGINT  YES  None    None  None\n8                 gsc_clicks      BIGINT  YES  None    None  None\n9           gsc_sum_position      BIGINT  YES  None    None  None\n10          gsc_avg_position      DOUBLE  YES  None    None  None\n11             ga4_pageviews      BIGINT  YES  None    None  None\n12              ga4_sessions      BIGINT  YES  None    None  None\n13                 ga4_users      BIGINT  YES  None    None  

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one (client_hash_id, content_hash_id, report_date) combination —
i.e., one content page's search + analytics performance snapshot for one client,
on one specific day.

Tables used: fact_content_daily_performance (main table),
partitioned by month (e.g., month=2026-03)

Time window: development month = 2026-03 (mid-panel),
final month (June 2026, in _sample) reserved as sealed test month

In [56]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

Sort every field you plan to touch into these four buckets. Excluded needs a why.



Feature columns (predictive signals, known at decision time):
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- scroll_events

Label (what we want to predict — a proxy):
- Whether a page's clicks will decline in the next period
  (derived by comparing future gsc_clicks to current gsc_clicks —
  NOT available at decision time, this is what we're trying to predict)

Context (identifiers, used for grouping/joining, not fed as predictive features):
- client_hash_id
- content_hash_id
- report_date
- month

Excluded (deliberately not used in this lane):
- ga4_* columns (pageviews, sessions, users) and ai_* AI-referral columns —
  excluded because this lane focuses on core Google Search (GSC) performance only,
  not on-site analytics or AI-referral traffic (that belongs to a different lane)

In [57]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
con.sql(f"""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 10
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  MIN(report_date) AS earliest_date,
  MAX(report_date) AS latest_date
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

In [ ]:
con.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5 Features (max 5) — with "available at decision time" reasoning

In [49]:
df = con.sql(f"""
SELECT
  client_hash_id,
  content_hash_id,
  report_date,
  gsc_impressions,
  gsc_clicks,
  CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE 0 END AS ctr,
  gsc_avg_position,
  scroll_events
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
""").df()

df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,scroll_events
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,0.000000,3.350000,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,0.000000,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,0.008000,4.928000,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,0.000000,4.000000,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,0.000000,2.272727,<NA>
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,0.004184,7.347280,<NA>
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,0.000000,7.832461,<NA>
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,0.000000,3.272727,<NA>
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,0,0.000000,5.636364,<NA>
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,0,0.000000,4.500000,<NA>


Feature reasoning (why each is knowable at the decision moment):

- gsc_impressions: logged by Google Search Console as of that report_date —
  already recorded before any decision is made.
- gsc_clicks: same — daily GSC log, available same-day.
- ctr (clicks/impressions): purely derived from the two fields above,
  both already available at decision time.
- gsc_avg_position: daily GSC ranking snapshot, historical/current — not future.
- scroll_events: on-page engagement signal logged for that day, available
  same-day as report_date.

## The Trap: deliberate leakage experiment

In [50]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd

# Build a simple future-looking label: did clicks drop vs a lagged value?
df2 = con.sql(f"""
SELECT
  client_hash_id, content_hash_id, report_date,
  gsc_impressions, gsc_clicks, gsc_avg_position, scroll_events,
  LAG(gsc_clicks, 1) OVER (PARTITION BY client_hash_id, content_hash_id ORDER BY report_date) AS prev_clicks
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
""").df()

df2 = df2.dropna(subset=['prev_clicks'])
df2['label'] = (df2['gsc_clicks'] < df2['prev_clicks']).astype(int)

# HONEST features (no leakage)
X_honest = df2[['gsc_impressions', 'gsc_avg_position', 'scroll_events']].fillna(0)
y = df2['label']

Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
honest_score = roc_auc_score(yte, model.predict_proba(Xte)[:,1])
print("HONEST score:", honest_score)

# --- THE TRAP: add a column derived FROM the label itself ---
df2['leaky_feature'] = df2['gsc_clicks']  # this directly determines the label!
X_leaky = df2[['gsc_impressions', 'gsc_avg_position', 'scroll_events', 'leaky_feature']].fillna(0)

Xtr, Xte, ytr, yte = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
leaky_score = roc_auc_score(yte, model.predict_proba(Xte)[:,1])
print("LEAKY score (with label-derived column):", leaky_score)

print("\n--- Conclusion ---")
print(f"Honest score: {honest_score:.3f} | Leaky score: {leaky_score:.3f}")
print("The leaky column caused an artificial jump because it directly encodes the label. Removing it and keeping the honest score.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

HONEST score: 0.7448586774461318
LEAKY score (with label-derived column): 0.7527109115937397

--- Conclusion ---
Honest score: 0.745 | Leaky score: 0.753
The leaky column caused an artificial jump because it directly encodes the label. Removing it and keeping the honest score.


Query 1 (Grain check): 0 duplicate rows found — confirms one row =
one (client, content, date) combination, as stated in Section 1.

Query 2 (Row count + date span): 9,841,378 rows, spanning 2026-03-01
to 2026-03-31 — full month coverage confirmed.

Query 3 (Availability, IS TRUE): Of 9,841,378 total rows, only
3,611,061 (~37%) have gsc_data_available IS TRUE. This means ~63%
of rows have missing/unavailable GSC data and must be filtered
out before use.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [51]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This slice only covers March 2026 (one month) — seasonal patterns and
longer-term trends cannot be observed. Additionally, only ~37% of rows
have gsc_data_available IS TRUE — the majority of rows have missing GSC
data, meaning this slice may be biased toward clients/pages with more
complete tracking setups, not representative of all pages equally.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.